In [1]:
# Convert PM2.5 data into xarray format

In [2]:
import os
import pyreadr

In [3]:
# Load Rdata file
PM_DIR = "/glade/work/awells/air_quality/PM2.5_obs/"
file = "GBD2016_PREDPOP_FINAL.RData"
file_path = os.path.join(PM_DIR, file)
data = pyreadr.read_r(file_path)

# Extract your dataframe
df = data["mydata2"]

In [4]:
# Step 1. Select PM2.5 columns (all years, all stats)
pm25_cols = [c for c in df.columns if "PM2.5" in c]

# Step 2. Melt to long format
df_long = df.melt(
    id_vars=["Latitude", "Longitude", "idgridcell", "Country", "region", "urban", "perurban"],
    value_vars=pm25_cols,
    var_name="variable",
    value_name="value"
)

# Step 3. Extract year and statistic type from column names
# e.g. "Median_PM2.5_2016" -> ("Median", 2016)
df_long["stat"] = df_long["variable"].str.extract(r"^(.*?)_PM2\.5")
df_long["year"] = df_long["variable"].str.extract(r"(\d{4})").astype(int)

# Step 4. Pivot so that stats become separate variables
df_wide = df_long.pivot_table(
    index=["year", "Latitude", "Longitude"],
    columns="stat",
    values="value"
).reset_index()

# Step 5. Convert to xarray
ds = df_wide.set_index(["year", "Latitude", "Longitude"]).to_xarray()

In [10]:
out_file = "DIMAQ_PM25_1990-2016.nc"
out_path = os.path.join(PM_DIR, out_file)
print(f"Saving to {out_path}")
ds.to_netcdf(out_path)

Saving to /glade/work/awells/air_quality/PM2.5_obs/DIMAQ_PM25_1990-2016.nc
